In [1]:
cd "C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph"

C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph


## CLEANER

#### Creating cleaner.py File

#### Testing Cleaner.py file

# — Transcript Preprocessing

##  — Normalize Transcript

In [17]:
from pathlib import Path 
from src.config.config import load_config
from src.data_ingestion.loader import TranscriptLoader
from src.preprocessing.cleaner import TranscriptCleaner

In [19]:
#load Transcript

config=load_config()

TRANSCRIPT_PATH=(Path(config["paths"]["raw_data"])/"Meeting_Transcript.txt")

document= TranscriptLoader.load_document(str(TRANSCRIPT_PATH))
                 

2026-09-07 11:09:10,482 - INFO - NumExpr defaulting to 8 threads.
2026-09-07 11:09:10 | INFO | src.data_ingestion.loader | Transcript loaded successfully: Meeting_Transcript.txt


In [20]:
#### Normalize Transcript

clean_document= TranscriptCleaner.clean(document)

2026-09-07 11:10:19 | INFO | src.preprocessing.cleaner | Transcript Normalized Sucessfully


In [26]:
print(clean_document.text[:500])

Product Manager: Good morning everyone. We have received several complaints about the mobile app crashing during login.

Customer Support Lead: Yes, our support team received more than 120 complaints in the last two days.

Product Manager: That is quite serious. We need to identify the root cause quickly.

Mobile Developer: I checked some of the crash logs yesterday and noticed that most of the issues are coming from Android users.

QA Tester: I also tried reproducing the issue on an And


In [27]:
print("Original\n")
print(document.text[:300])

print("\n" + "=" * 80 + "\n")

print("Normalized\n")
print(clean_document.text[:300])

Original

Product Manager: Good morning everyone. We have received several complaints about the mobile app crashing during login.

Customer Support Lead: Yes, our support team received more than 120 complaints in the last two days.

Product Manager: That is quite serious. We need to identify the root caus


Normalized

Product Manager: Good morning everyone. We have received several complaints about the mobile app crashing during login.

Customer Support Lead: Yes, our support team received more than 120 complaints in the last two days.

Product Manager: That is quite serious. We need to identify the root caus


#### Verify MetaData

In [28]:
clean_document.metadata

{'file_path': 'data\\raw\\Meeting_Transcript.txt',
 'file_name': 'Meeting_Transcript.txt',
 'file_type': 'text/plain',
 'file_size': 3917,
 'creation_date': '2026-09-04',
 'last_modified_date': '2026-09-02'}

##  TransCript Speaker Parsing

# Transcript Parsing 

In [35]:
from src.preprocessing.speaker_parser import SpeakerParser

In [37]:
speaker_messages= SpeakerParser.parse(clean_document)

2026-09-07 11:44:43 | INFO | src.preprocessing.speaker_parser | Parsed 47 speaker messages.


In [40]:
speaker_messages[:10]

[{'speaker': 'Product Manager',
  'message': 'Good morning everyone. We have received several complaints about the mobile app crashing during login.'},
 {'speaker': 'Customer Support Lead',
  'message': 'Yes, our support team received more than 120 complaints in the last two days.'},
 {'speaker': 'Product Manager',
  'message': 'That is quite serious. We need to identify the root cause quickly.'},
 {'speaker': 'Mobile Developer',
  'message': 'I checked some of the crash logs yesterday and noticed that most of the issues are coming from Android users.'},
 {'speaker': 'QA Tester',
  'message': 'I also tried reproducing the issue on an Android device and the app crashed right after entering login credentials.'},
 {'speaker': 'Backend Developer',
  'message': 'Could the issue be related to the authentication API?'},
 {'speaker': 'Mobile Developer',
  'message': 'That is possible. The login request might be failing due to some recent backend changes.'},
 {'speaker': 'Product Manager',
  'm

In [39]:
len(speaker_messages)

47

In [42]:
import pandas as pd

pd.DataFrame(speaker_messages)

,speaker,message
0,Product Manager,Good morning everyone. We have received severa...
1,Customer Support Lead,"Yes, our support team received more than 120 c..."
2,Product Manager,That is quite serious. We need to identify the...
3,Mobile Developer,I checked some of the crash logs yesterday and...
4,QA Tester,I also tried reproducing the issue on an Andro...
5,Backend Developer,Could the issue be related to the authenticati...
6,Mobile Developer,That is possible. The login request might be f...
7,Product Manager,When was the last update deployed to production?
8,Backend Developer,We deployed a small update to the authenticati...
9,QA Tester,The crash reports also started appearing aroun...


## CHUNKING

#### Creating Chunking.py file

%%writefile src/preprocessing/chunker.py

"""
Transcript Chunker Module

Phase 2 — Preprocessing

Creates LlamaIndex Nodes from meeting transcripts.
"""

import sys

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document, TextNode

from src.utils.exception import ProjectException
from src.utils.logger import get_logger

logger = get_logger(__name__)


class TranscriptChunker:
    

    CHUNK_SIZE = 512
    CHUNK_OVERLAP = 50

    @classmethod
    def create_nodes(cls, document: Document) -> list[TextNode]:
       
        try:
            splitter = SentenceSplitter(
                chunk_size=cls.CHUNK_SIZE,
                chunk_overlap=cls.CHUNK_OVERLAP,
            )

            nodes = splitter.get_nodes_from_documents([document])

            logger.info(f"Created {len(nodes)} transcript chunks.")

            return nodes

        except Exception as error:
            logger.error(str(error))
            raise ProjectException(str(error), sys)



### 3 Testing Chunking.py file

!pytest tests/test_chunker.py -v

In [53]:
from src.preprocessing.chunker import TranscriptChunker

In [54]:
nodes=TranscriptChunker.create_nodes(clean_document)
len(nodes)

2026-09-07 13:22:12 | INFO | src.preprocessing.chunker | Created 2 transcript chunks.


2

In [57]:
first_node=nodes[0]
type(first_node)

llama_index.core.schema.TextNode

In [58]:
print(first_node.text)

Product Manager: Good morning everyone. We have received several complaints about the mobile app crashing during login.

Customer Support Lead: Yes, our support team received more than 120 complaints in the last two days.

Product Manager: That is quite serious. We need to identify the root cause quickly.

Mobile Developer: I checked some of the crash logs yesterday and noticed that most of the issues are coming from Android users.

QA Tester: I also tried reproducing the issue on an Android device and the app crashed right after entering login credentials.

Backend Developer: Could the issue be related to the authentication API?

Mobile Developer: That is possible. The login request might be failing due to some recent backend changes.

Product Manager: When was the last update deployed to production?

Backend Developer: We deployed a small update to the authentication service three days ago.

QA Tester: The crash reports also started appearing around the same time.

Mobile Developer: 

In [59]:
first_node.metadata

{'file_path': 'data\\raw\\Meeting_Transcript.txt',
 'file_name': 'Meeting_Transcript.txt',
 'file_type': 'text/plain',
 'file_size': 3917,
 'creation_date': '2026-09-04',
 'last_modified_date': '2026-09-02'}

In [60]:
for index, node in enumerate(nodes[:5], start=1):
    print(f"Chunk {index}")
    print(f"Characters : {len(node.text)}")
    print("-" * 50)

Chunk 1
Characters : 2324
--------------------------------------------------
Chunk 2
Characters : 1798
--------------------------------------------------
